# 원룸 인페인팅 + 가구 On/Off 통합 파이프라인 (v2.0)

**핵심 설계**: DUSt3R 재구성을 **한 번만** 돌려서 좌표계를 통일한다.

- 원본 이미지로 재구성 → 가구별 메시 분리 (v4.7 방식, 토글 가능)
- 가구 자리에 생기는 구멍(backdrop)은 **LaMa로 인페인팅한 이미지를 각 카메라 시점에 재투영**해서 색을 입힘
  - 기존 v4.7 backdrop은 "근처 실제 바닥/벽 점들의 평균색"으로 채워서 그림자/이음새가 어색했음
  - 이번엔 실제로 가구가 있던 그 픽셀 위치에 LaMa가 그려넣은 색을 그대로 가져다 쓰므로 훨씬 자연스러움
- 좌표계가 하나라 가구 On일 때(원본 텍스처)와 Off일 때(인페인팅 backdrop)가 정확히 같은 자리에 맞물림
- three.js 뷰어에서 가구별 체크박스로 On/Off 토글

**사전 조건**: v4.7에서 세그 라벨을 `seg_save()`로 저장해 둔 상태 (`{TARGET_FOLDER}_seg_labels.npz/.json`)


## 0. 환경 설정

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os, sys, glob, json, io, base64
import numpy as np
from PIL import Image

if not os.path.exists("/content/dust3r"):
    !git clone --recursive https://github.com/naver/dust3r.git /content/dust3r

%cd /content/dust3r
!pip install -q -r requirements.txt
!pip install -q open3d shapely trimesh scipy roma fast_simplification
%cd /content

if "/content/dust3r" not in sys.path:
    sys.path.insert(0, "/content/dust3r")

import cv2
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")


Cloning into '/content/dust3r'...
remote: Enumerating objects: 611, done.
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)
Receiving objects: 100% (611/611), 756.60 KiB | 39.82 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule 'croco' (https://github.com/naver/croco) registered for path 'croco'
Cloning into '/content/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects: 100% (87/87), done.        
remote: Compressing objects: 100% (54/54), done.        
remote: Total 198 (delta 54), reused 33 (delta 33), pack-reused 111 (from 1)        
Receiving objects: 100% (198/198), 403.93 KiB | 8.98 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Submodule path 'croco': checked out 'd7de0705845239092414480bd829228723bf20de'
/content/dust3r
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# LaMa JIT 모델 (pip 패키지 대신 직접 로드 - 의존성 충돌 회피)
LAMA_PT = "/content/big-lama.pt"
if not os.path.exists(LAMA_PT):
    torch.hub.download_url_to_file(
        "https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt",
        LAMA_PT,
    )
lama_model = torch.jit.load(LAMA_PT, map_location=device)
lama_model.eval()
print("LaMa 모델 로드 완료")


100%|██████████| 196M/196M [00:02<00:00, 90.0MB/s]


LaMa 모델 로드 완료


## 1. 데이터 + 저장된 세그 마스크 로드

In [4]:
TARGET_FOLDER = "room06"
CONF_THR = 1.0

# 인페인팅(=제거) 대상 라벨. v4.7 TARGET_LABELS와 동일
TARGET_LABELS = {"bed", "refrigerator", "chair", "wardrobe", "desk", "잡동사니"}

DATA_ROOT = "/content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data"
target_path = f"{DATA_ROOT}/raw_room/{TARGET_FOLDER}"

img_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    img_paths.extend(glob.glob(os.path.join(target_path, ext)))
img_paths = sorted(img_paths)
images_pil = [Image.open(p).convert("RGB") for p in img_paths]

print(f"[{TARGET_FOLDER}] 이미지 {len(img_paths)}장")
for p in img_paths:
    print(" ", os.path.basename(p))


[room06] 이미지 10장
  img_01.jpg
  img_02.jpg
  img_03.jpg
  img_04.jpg
  img_05.jpg
  img_06.jpg
  img_07.jpg
  img_08.jpg
  img_09.jpg
  img_10.jpg


In [5]:
# v4.7 seg_save() 저장본 로드 -> all_results 구조로 복원 (이미지별 masks/boxes/labels)
SEG_SAVE_BASE = f"{DATA_ROOT}/{TARGET_FOLDER}_seg_labels"

with open(SEG_SAVE_BASE + ".json") as f:
    _seg_meta = json.load(f)
_seg_arrs = np.load(SEG_SAVE_BASE + ".npz")
_by_name = {m["filename"]: (mi, m) for mi, m in enumerate(_seg_meta["images"])}

all_results = []
for path in img_paths:
    name = os.path.basename(path)
    img = np.array(Image.open(path).convert("RGB"))
    masks, boxes, labels = [], [], []
    if name in _by_name:
        mi, m = _by_name[name]
        for k, label in enumerate(m["labels"]):
            masks.append(_seg_arrs[f"m_{mi}_{k}"].astype(bool))
            boxes.append(np.array(m["boxes"][k], dtype=np.float32))
            labels.append(label)
    else:
        print(f"[경고] 저장본에 없음: {name}")
    all_results.append({"path": path, "img": img, "masks": masks, "boxes": boxes, "labels": labels})
    print(f"{name}: {labels}")

# 프레임별 인페인팅(제거) 대상 union 마스크
furn_masks = []
for res in all_results:
    H, W = res["img"].shape[:2]
    union = np.zeros((H, W), dtype=bool)
    for m, lb in zip(res["masks"], res["labels"]):
        if lb in TARGET_LABELS:
            union |= m
    furn_masks.append(union)


img_01.jpg: ['bed']
img_02.jpg: ['bed']
img_03.jpg: []
img_04.jpg: ['bed']
img_05.jpg: ['refrigerator']
img_06.jpg: []
img_07.jpg: []
img_08.jpg: ['bed']
img_09.jpg: ['bed']
img_10.jpg: ['bed']


## 2. DUSt3R 재구성 (원본 이미지, 1회만)

In [6]:
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.inference import inference
from dust3r.cloud_opt import global_aligner

model = AsymmetricCroCo3DStereo.from_pretrained(
    "naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt"
).to(device)
model.eval()

images_dust3r = load_images(img_paths, size=512)
pairs = make_pairs(images_dust3r, scene_graph="complete", prefilter=None, symmetrize=True)
print(f"이미지 쌍 {len(pairs)}개 구성")

with torch.no_grad():
    output = inference(pairs, model, device, batch_size=2)

scene = global_aligner(output, device=device)
scene.compute_global_alignment(niter=300, init="mst")
print("3D 복원 및 정렬 완료")


Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

>> Loading a list of 10 images
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_01.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_02.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_03.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_04.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_05.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_06.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_07.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputer

  0%|          | 0/45 [00:00<?, ?it/s]/content/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 45/45 [00:20<00:00,  2.21it/s]


 init edge (0*,1*) score=np.float64(382.89166259765625)
 init edge (0,9*) score=np.float64(249.53294372558594)
 init edge (2*,1) score=np.float64(191.88235473632812)
 init edge (9,8*) score=np.float64(70.78853607177734)
 init edge (2,3*) score=np.float64(266.15899658203125)
 init edge (3,4*) score=np.float64(236.8054962158203)
 init edge (4,5*) score=np.float64(86.14830780029297)
 init edge (7*,5) score=np.float64(110.87826538085938)
 init edge (7,6*) score=np.float64(263.4967346191406)
 init loss = 0.015211939811706543
Global alignement - optimizing for:
['pw_poses', 'im_depthmaps', 'im_poses', 'im_focals']


  0%|          | 0/300 [00:00<?, ?it/s]/content/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), lr
100%|██████████| 300/300 [00:30<00:00,  9.76it/s, lr=1.27413e-06 loss=0.0101352]

3D 복원 및 정렬 완료


## 3. 바닥 정렬 (v4.7과 동일)

In [7]:
import trimesh
from scipy.spatial.transform import Rotation

try:
    from dust3r.demo import OPENGL
except ImportError:
    from dust3r.viz import OPENGL

cams2world = scene.get_im_poses().detach().cpu().numpy()
_rot = np.eye(4)
_rot[:3, :3] = Rotation.from_euler("y", np.deg2rad(180)).as_matrix()
GLB_TRANSFORM = np.linalg.inv(cams2world[0] @ OPENGL @ _rot)


def apply_transform(points, T):
    pts_h = np.concatenate([points, np.ones((len(points), 1))], axis=1)
    return (T @ pts_h.T).T[:, :3]


def find_floor_rotation(points, up_axis=1, n_iter=2000):
    height = points[:, up_axis]
    floor_candidates = points[height <= np.percentile(height, 35)]
    if len(floor_candidates) < 3:
        return np.eye(4), np.array([0.0, 1.0, 0.0])

    diag = np.linalg.norm(points.max(0) - points.min(0))
    threshold = diag * 0.02

    best_inliers, best_normal = -1, np.array([0.0, 1.0, 0.0])
    for _ in range(n_iter):
        idx = np.random.choice(len(floor_candidates), 3, replace=False)
        p1, p2, p3 = floor_candidates[idx]
        normal = np.cross(p2 - p1, p3 - p1)
        nrm = np.linalg.norm(normal)
        if nrm < 1e-6:
            continue
        normal = normal / nrm
        d = -np.dot(normal, p1)
        dist = np.abs(floor_candidates @ normal + d)
        ninl = int((dist < threshold).sum())
        if ninl > best_inliers:
            best_inliers, best_normal = ninl, normal

    if best_normal[up_axis] < 0:
        best_normal = -best_normal

    target = np.array([0.0, 1.0, 0.0])
    v = np.cross(best_normal, target)
    s = np.linalg.norm(v)
    c = float(np.dot(best_normal, target))
    if s < 1e-8:
        R = np.eye(3)
    else:
        vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c) / (s * s))

    T = np.eye(4)
    T[:3, :3] = R
    return T, best_normal


pointmaps = scene.get_pts3d()
confidences = scene.get_conf()

_all_pts_raw, _all_cols = [], []
for i, pts in enumerate(pointmaps):
    pts_np = pts.detach().cpu().numpy().reshape(-1, 3)
    conf_np = confidences[i].detach().cpu().numpy().flatten()
    m = conf_np > CONF_THR
    h, w = pts.shape[:2]
    img_resized = np.array(images_pil[i].resize((w, h))).reshape(-1, 3)
    _all_pts_raw.append(pts_np[m])
    _all_cols.append(img_resized[m])
_all_pts_raw = np.concatenate(_all_pts_raw, axis=0)
all_cols = np.concatenate(_all_cols, axis=0) / 255.0

_all_pts_glb = apply_transform(_all_pts_raw, GLB_TRANSFORM)
FLOOR_ROT, floor_normal = find_floor_rotation(_all_pts_glb)
print(f"검출된 바닥 법선(GLB좌표): [{floor_normal[0]:+.2f}, {floor_normal[1]:+.2f}, {floor_normal[2]:+.2f}]")

ALIGN_TRANSFORM = FLOOR_ROT @ GLB_TRANSFORM
ALIGN_INV = np.linalg.inv(ALIGN_TRANSFORM)
all_pts = apply_transform(_all_pts_raw, ALIGN_TRANSFORM)
print(f"정렬 완료. 전체 포인트 수: {len(all_pts):,}")


검출된 바닥 법선(GLB좌표): [+0.03, +0.98, -0.19]
정렬 완료. 전체 포인트 수: 1,948,128


## 4. 포인트별 라벨 부여 (backdrop 참조점 선별용)

In [8]:
point_labels = np.empty(len(all_pts), dtype=object)
point_labels[:] = ""

_offset = 0
for vi, pts in enumerate(pointmaps):
    h, w = pts.shape[:2]
    conf_np = confidences[vi].detach().cpu().numpy().flatten()
    conf_mask = conf_np > CONF_THR
    n_view = int(conf_mask.sum())

    res = all_results[vi]
    order = sorted(range(len(res["masks"])),
                   key=lambda k: (res["boxes"][k][2]-res["boxes"][k][0]) *
                                 (res["boxes"][k][3]-res["boxes"][k][1]),
                   reverse=True)

    view_label_flat = np.empty(h * w, dtype=object)
    view_label_flat[:] = ""
    for k in order:
        lbl = res["labels"][k]
        if lbl not in TARGET_LABELS:
            continue
        m_resized = cv2.resize(res["masks"][k].astype(np.uint8), (w, h),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
        view_label_flat[m_resized.flatten()] = lbl

    point_labels[_offset:_offset + n_view] = view_label_flat[conf_mask]
    _offset += n_view

assert _offset == len(all_pts)
uniq, cnts = np.unique(point_labels, return_counts=True)
for u, c in sorted(zip(uniq, cnts), key=lambda x: -x[1]):
    print(f"  {(u if u else '(미분류)'):15s}: {c:>7,}점")


  (미분류)          : 1,772,329점
  bed            : 147,497점
  refrigerator   :  28,302점


## 5. 천장고 기반 스케일 계산

In [9]:
CEILING_CANDIDATES = [2.3, 2.4]
UP_AXIS = 1
up = all_pts[:, UP_AXIS]

hist, edges = np.histogram(up, bins=80)
centers = (edges[:-1] + edges[1:]) / 2
n = len(centers)
floor_peak = centers[:int(n*0.35)][np.argmax(hist[:int(n*0.35)])]
ceil_peak  = centers[int(n*0.65):][np.argmax(hist[int(n*0.65):])]
ceiling_raw = ceil_peak - floor_peak

SCALE = CEILING_CANDIDATES[0] / ceiling_raw
print(f"바닥 Y={floor_peak:.4f}, 천장 Y={ceil_peak:.4f}, SCALE={SCALE:.4f} (천장고 {CEILING_CANDIDATES[0]}m 기준)")


바닥 Y=-0.1436, 천장 Y=0.0922, SCALE=9.7548 (천장고 2.3m 기준)


## 6. LaMa 인페인팅 (프레임별 가구 제거)

In [10]:
!pip install -q diffusers transformers accelerate

from diffusers import StableDiffusionInpaintPipeline
import torch

sd_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
).to(device)

def sd_inpaint(img_np, mask_bool, prompt="empty tile floor, seamless, photorealistic, matching room lighting"):
    from PIL import Image
    H, W = mask_bool.shape
    img_pil = Image.fromarray(img_np).resize((512, 512))
    mask_pil = Image.fromarray((mask_bool * 255).astype(np.uint8)).resize((512, 512))

    result = sd_pipe(
        prompt=prompt,
        negative_prompt="furniture, bed, blurry, distorted",
        image=img_pil,
        mask_image=mask_pil,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]

    return np.array(result.resize((W, H)))

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [11]:
def hybrid_inpaint(img_np, mask_bool,
                    refine_strength=0.35,   # 낮을수록 LaMa 결과에 더 가깝게(물건 생성 억제). 0.25~0.45 사이 튜닝
                    guidance_scale=4.0):
    from PIL import Image

    H, W = mask_bool.shape

    # 1단계: LaMa로 물건 없는 밑그림 (기존 함수 재사용)
    lama_base = lama_inpaint(img_np, mask_bool)

    # 2단계: SD img2img로 마스크 영역만 텍스처 다듬기 (낮은 strength)
    scale = 512 / max(H, W)
    new_h, new_w = (int(H * scale) // 8) * 8, (int(W * scale) // 8) * 8

    base_pil = Image.fromarray(lama_base).resize((new_w, new_h))
    mask_pil = Image.fromarray((mask_bool * 255).astype(np.uint8)).resize((new_w, new_h))

    result = sd_pipe(
        prompt="plain wall and floor texture, seamless surface, no objects, empty",
        negative_prompt=("furniture, bed, chair, sofa, rug, carpet, pillow, cushion, "
                          "table, object, item, decoration, colorful shape, pattern, "
                          "person, animal, box, container"),
        image=base_pil,
        mask_image=mask_pil,
        strength=refine_strength,
        num_inference_steps=30,
        guidance_scale=guidance_scale,
    ).images[0]

    return np.array(result.resize((W, H)))

In [12]:
DILATE_PX = 15  # 그림자 잔상 남으면 25~35로 올려볼 것

def lama_inpaint(img_np, mask_bool):
    H, W = mask_bool.shape
    img_t = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    mask_t = torch.from_numpy(mask_bool.astype(np.float32))[None, None]

    pad_h = (8 - H % 8) % 8
    pad_w = (8 - W % 8) % 8
    img_t = torch.nn.functional.pad(img_t, (0, pad_w, 0, pad_h), mode="reflect")
    mask_t = torch.nn.functional.pad(mask_t, (0, pad_w, 0, pad_h), mode="reflect")
    mask_t = (mask_t > 0).float()

    with torch.no_grad():
        out = lama_model(img_t.to(device), mask_t.to(device))

    out_np = out[0].permute(1, 2, 0).cpu().numpy()[:H, :W]
    return np.clip(out_np * 255, 0, 255).astype(np.uint8)


CROP_MARGIN = 2.0  # 마스크 박스 크기의 배수로 여유를 둠. 1.5~2.5 사이로 튜닝

def lama_inpaint_cropped(img_np, mask_bool, margin=CROP_MARGIN, target_long=768):
    H, W = img_np.shape[:2]
    ys, xs = np.where(mask_bool)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    bh, bw = y1 - y0, x1 - x0

    cy0 = max(0, int(y0 - bh * margin / 2))
    cy1 = min(H, int(y1 + bh * margin / 2))
    cx0 = max(0, int(x0 - bw * margin / 2))
    cx1 = min(W, int(x1 + bw * margin / 2))

    crop_img = img_np[cy0:cy1, cx0:cx1]
    crop_mask = mask_bool[cy0:cy1, cx0:cx1]

    # 크롭 영역도 너무 크면 다운스케일 (LaMa는 512~1024 근방이 안정적)
    ch, cw = crop_img.shape[:2]
    scale = min(1.0, target_long / max(ch, cw))
    if scale < 1.0:
        small_img = cv2.resize(crop_img, (int(cw*scale), int(ch*scale)), interpolation=cv2.INTER_AREA)
        small_mask = cv2.resize(crop_mask.astype(np.uint8), (int(cw*scale), int(ch*scale)),
                                interpolation=cv2.INTER_NEAREST).astype(bool)
    else:
        small_img, small_mask = crop_img, crop_mask

    result_small = lama_inpaint(small_img, small_mask)  # 기존 함수 재사용

    if scale < 1.0:
        result_crop = cv2.resize(result_small, (cw, ch), interpolation=cv2.INTER_LANCZOS4)
    else:
        result_crop = result_small

    # 마스크 경계 부드럽게 합성 (원본 크롭과 결과 크롭을 블렌딩)
    mask_f = cv2.GaussianBlur(crop_mask.astype(np.float32), (21, 21), 0)[..., None]
    result_crop = (result_crop * mask_f + crop_img * (1 - mask_f)).astype(np.uint8)

    out = img_np.copy()
    out[cy0:cy1, cx0:cx1] = result_crop
    return out




In [13]:

INPAINT_DIR = f"/content/inpainted/{TARGET_FOLDER}"
os.makedirs(INPAINT_DIR, exist_ok=True)
_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (DILATE_PX * 2 + 1,) * 2)

images_inpainted_pil = []
for path, mask in zip(img_paths, furn_masks):
    img = np.array(Image.open(path).convert("RGB"))
    if mask.any():
        mask_d = cv2.dilate(mask.astype(np.uint8), _kernel).astype(bool)
        result = hybrid_inpaint(img, mask_d) ####여기 인페인팅 결과 함수 수정
    else:
        result = img
    out_path = os.path.join(INPAINT_DIR, os.path.splitext(os.path.basename(path))[0] + ".png")
    Image.fromarray(result).save(out_path)
    images_inpainted_pil.append(Image.fromarray(result))
    print(f"{os.path.basename(path)}: {'인페인팅 완료' if mask.any() else '건너뜀'}")

  0%|          | 0/10 [00:00<?, ?it/s]

img_01.jpg: 인페인팅 완료


  0%|          | 0/10 [00:00<?, ?it/s]

img_02.jpg: 인페인팅 완료
img_03.jpg: 건너뜀


  0%|          | 0/10 [00:00<?, ?it/s]

...

img_08.jpg: 인페인팅 완료


  0%|          | 0/10 [00:00<?, ?it/s]

img_09.jpg: 인페인팅 완료


  0%|          | 0/10 [00:00<?, ?it/s]

img_10.jpg: 인페인팅 완료


In [14]:
# 전/후 비교 (넘어가기 전에 품질 확인)
import matplotlib.pyplot as plt

n = len(img_paths)
fig, axes = plt.subplots(2, n, figsize=(6 * n, 11))
if n == 1:
    axes = axes.reshape(2, 1)
for i, (orig_p, inp_img, mask) in enumerate(zip(img_paths, images_inpainted_pil, furn_masks)):
    axes[0, i].imshow(np.array(Image.open(orig_p).convert("RGB")))
    axes[0, i].set_title(f"원본 | 마스크 {mask.sum():,}px", fontsize=10)
    axes[0, i].axis("off")
    axes[1, i].imshow(np.array(inp_img))
    axes[1, i].set_title("LaMa 인페인팅", fontsize=10)
    axes[1, i].axis("off")
plt.suptitle(f"[{TARGET_FOLDER}] 가구 제거 인페인팅 전/후", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.

## 7. 방/가구 메시 분리 (원본 텍스처, v4.7 방식)

In [ ]:
from dust3r.demo import pts3d_to_trimesh, cat_meshes
from dust3r.utils.device import to_numpy


def build_room_meshes(scene, all_results, target_labels, min_conf_thr=3, clean_depth=True):
    if clean_depth:
        scene = scene.clean_pointcloud()

    rgbimg = scene.imgs
    pts3d = to_numpy(scene.get_pts3d())
    scene.min_conf_thr = float(scene.conf_trf(torch.tensor(min_conf_thr)))
    msk = [np.asarray(m).astype(bool) for m in to_numpy(scene.get_masks())]
    n_img = len(rgbimg)

    furn_union, furn_per_label = [], []
    for i in range(n_img):
        H, W = msk[i].shape[:2]
        union = np.zeros((H, W), bool)
        per = {}
        res = all_results[i]
        for j, label in enumerate(res["labels"]):
            if label not in target_labels:
                continue
            m = cv2.resize(res["masks"][j].astype(np.uint8), (W, H),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
            per[label] = per.get(label, np.zeros((H, W), bool)) | m
            union |= m
        furn_union.append(union)
        furn_per_label.append(per)

    all_labels = sorted({lab for per in furn_per_label for lab in per})

    def _meshes_from_valid(valid_list):
        pieces = []
        for i in range(n_img):
            if valid_list[i].sum() < 3:
                continue
            piece = pts3d_to_trimesh(rgbimg[i], pts3d[i], valid_list[i])
            if len(piece["faces"]):
                pieces.append(piece)
        if not pieces:
            return None
        mesh = trimesh.Trimesh(**cat_meshes(pieces))
        mesh.apply_transform(ALIGN_TRANSFORM)
        return mesh

    out = {}
    room_valid = [msk[i] & ~furn_union[i] for i in range(n_img)]
    out["room"] = _meshes_from_valid(room_valid)

    for label in all_labels:
        obj_valid = [msk[i] & furn_per_label[i].get(label, np.zeros_like(msk[i], dtype=bool))
                     for i in range(n_img)]
        m_obj = _meshes_from_valid(obj_valid)
        if m_obj is not None:
            out[label] = m_obj

    return out


room_parts = build_room_meshes(scene, all_results, TARGET_LABELS, min_conf_thr=3, clean_depth=True)
for name, m in room_parts.items():
    if m is not None:
        print(f"  {name:14s}: 정점 {len(m.vertices):>7,}개 / 면 {len(m.faces):>7,}개")


## 8. Backdrop 지오메트리 (가구 뒤 바닥/벽 슬래브)

외곽선 추출 → 벽/바닥 메시 직접 구성까지는 v4.7과 동일. 색상만 다음 셀에서 LaMa 재투영으로 바꾼다.

In [ ]:
from scipy import ndimage
from scipy.spatial import cKDTree
from shapely.geometry import Polygon as _ShapelyPolygon
from shapely.ops import unary_union

WALL_STRATEGY = "full_height"
BD_CELL       = 0.05
BD_FLOOR_BAND = (-0.03, 0.12)
BD_CLOSE_IT   = 3
BD_SMOOTH_IT  = 2
BD_APPROX_EPS = 0.008
BD_PUSH       = 0.03 / SCALE
BD_SUBDIV_M   = 0.10
WALL_ZONE_TOP_MARGIN = 0.05 / SCALE
FURN_UNION_MARGIN    = 0.05 / SCALE

_is_bg = (point_labels == "")
_y = all_pts[:, 1]
_contour_mask = _is_bg & (_y > floor_peak + BD_FLOOR_BAND[0]) & (_y < ceil_peak - WALL_ZONE_TOP_MARGIN)
_floor_xz_raw = all_pts[_contour_mask][:, [0, 2]]

_floor_xz_m = _floor_xz_raw * SCALE
_xmin, _zmin = _floor_xz_m.min(0) - BD_CELL * 3
_xmax, _zmax = _floor_xz_m.max(0) + BD_CELL * 3
_GW = int(np.ceil((_xmax - _xmin) / BD_CELL))
_GH = int(np.ceil((_zmax - _zmin) / BD_CELL))

_gx = np.clip(((_floor_xz_m[:, 0] - _xmin) / BD_CELL).astype(int), 0, _GW - 1)
_gz = np.clip(((_floor_xz_m[:, 1] - _zmin) / BD_CELL).astype(int), 0, _GH - 1)
_occ = np.zeros((_GH, _GW), dtype=bool)
_occ[_gz, _gx] = True

_occ = ndimage.binary_closing(_occ, iterations=BD_CLOSE_IT)
_occ = ndimage.binary_fill_holes(_occ)
_lab, _nc = ndimage.label(_occ)
if _nc > 1:
    _sizes = ndimage.sum(np.ones_like(_lab), _lab, range(1, _nc + 1))
    _occ = (_lab == int(np.argmax(_sizes)) + 1)
_occ = ndimage.binary_opening(_occ, iterations=BD_SMOOTH_IT)
_occ = ndimage.binary_closing(_occ, iterations=BD_SMOOTH_IT)
_occ = ndimage.binary_fill_holes(_occ)

_mask_u8 = (_occ.astype(np.uint8)) * 255
_contours, _ = cv2.findContours(_mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
_cnt = max(_contours, key=cv2.contourArea)
_peri = cv2.arcLength(_cnt, True)
_approx = cv2.approxPolyDP(_cnt, BD_APPROX_EPS * _peri, True)

_poly_grid = _approx[:, 0, :].astype(float)
_poly_m = np.stack([
    _xmin + (_poly_grid[:, 0] + 0.5) * BD_CELL,
    _zmin + (_poly_grid[:, 1] + 0.5) * BD_CELL,
], axis=1)
_poly_raw = _poly_m / SCALE

_shp = _ShapelyPolygon(_poly_raw)
if not _shp.is_valid:
    _shp = _shp.buffer(0)

_furn_labels_present = [u for u in np.unique(point_labels) if u]
_furn_polys = []
for _lbl in _furn_labels_present:
    _fp = all_pts[point_labels == _lbl][:, [0, 2]]
    if len(_fp) < 10:
        continue
    _hull = cv2.convexHull(_fp.astype(np.float32))[:, 0, :]
    _fpoly = _ShapelyPolygon(_hull).buffer(FURN_UNION_MARGIN)
    if _fpoly.is_valid and _fpoly.area > 0:
        _furn_polys.append(_fpoly)
if _furn_polys:
    _shp = unary_union([_shp] + _furn_polys)
    if _shp.geom_type == "MultiPolygon":
        _shp = max(_shp.geoms, key=lambda g: g.area)

_shp_pushed = _shp.buffer(BD_PUSH, join_style=2)
if _shp_pushed.geom_type == "MultiPolygon":
    _shp_pushed = max(_shp_pushed.geoms, key=lambda g: g.area)

_ring = np.asarray(_shp_pushed.exterior.coords)[:-1]
_K = len(_ring)
_y_floor = floor_peak - BD_PUSH
_y_ceil  = ceil_peak + BD_PUSH

_wall_v, _wall_f = [], []
for k in range(_K):
    x0, z0 = _ring[k]
    x1, z1 = _ring[(k + 1) % _K]
    base = len(_wall_v)
    _wall_v += [[x0, _y_floor, z0], [x1, _y_floor, z1],
                [x1, _y_ceil,  z1], [x0, _y_ceil,  z0]]
    _wall_f += [[base, base + 1, base + 2], [base, base + 2, base + 3]]
_walls = trimesh.Trimesh(vertices=np.array(_wall_v, float), faces=np.array(_wall_f, int), process=False)


def _ear_clip(poly_xz):
    n = len(poly_xz)
    if n < 3:
        return []
    area2 = sum(poly_xz[i][0]*poly_xz[(i+1)%n][1] - poly_xz[(i+1)%n][0]*poly_xz[i][1] for i in range(n))
    idx = list(range(n))

    def _cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

    def _in_tri(p, a, b, c):
        d1, d2, d3 = _cross(a,b,p), _cross(b,c,p), _cross(c,a,p)
        neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
        pos = (d1 > 0) or (d2 > 0) or (d3 > 0)
        return not (neg and pos)

    tris = []
    guard = 0
    while len(idx) > 3 and guard < 10000:
        guard += 1
        ear = False
        m = len(idx)
        for i in range(m):
            ia, ib, ic = idx[(i-1) % m], idx[i], idx[(i+1) % m]
            a, b, c = poly_xz[ia], poly_xz[ib], poly_xz[ic]
            if _cross(a, b, c) <= 0:
                continue
            bad = any(_in_tri(poly_xz[j], a, b, c) for j in idx if j not in (ia, ib, ic))
            if bad:
                continue
            tris.append((ia, ib, ic))
            del idx[i]
            ear = True
            break
        if not ear:
            break
    if len(idx) == 3:
        tris.append((idx[0], idx[1], idx[2]))
    return tris


_tris = _ear_clip(_ring)
if len(_tris) >= _K - 2:
    _floor_v = np.column_stack([_ring[:, 0], np.full(_K, _y_floor), _ring[:, 1]])
    _floor_f = np.array(_tris, int)
else:
    _cen = _ring.mean(0)
    _floor_v = np.vstack([[_cen[0], _y_floor, _cen[1]],
                          np.column_stack([_ring[:, 0], np.full(_K, _y_floor), _ring[:, 1]])])
    _floor_f = np.array([[0, 1 + k, 1 + (k + 1) % _K] for k in range(_K)], int)
_slab = trimesh.Trimesh(vertices=_floor_v, faces=_floor_f, process=False)

_backdrop = trimesh.util.concatenate([_walls, _slab])
_bd_v_sub, _bd_f_sub = trimesh.remesh.subdivide_to_size(
    _backdrop.vertices, _backdrop.faces, max_edge=BD_SUBDIV_M / SCALE)
_backdrop = trimesh.Trimesh(vertices=_bd_v_sub, faces=_bd_f_sub, process=False)
print(f"backdrop 지오메트리: 정점 {len(_backdrop.vertices):,} / 면 {len(_backdrop.faces):,}")


## 9. Backdrop 색상 = LaMa 인페인팅 이미지 재투영

핵심 아이디어: backdrop 정점을 각 카메라 시점으로 역투영해서, 그 픽셀 위치의 **인페인팅된 이미지 색**을 그대로 가져온다.
가구가 있던 자리는 정확히 그 카메라에서 LaMa가 채워넣은 색이라 그림자/질감이 훨씬 자연스럽다.
여러 뷰에서 겹치면 중앙값으로 합성 (한 뷰만 오염돼도 강건하게).

In [ ]:
def reproject_colors(verts_aligned, cams2world, focals, pps, pointmaps, images_list, align_inv):
    """backdrop 정점 -> 각 카메라 시점 재투영 -> 해당 이미지 픽셀 색 샘플링(뷰별 중앙값)"""
    n_v = len(verts_aligned)
    n_views = len(cams2world)
    verts_raw = apply_transform(verts_aligned, align_inv)

    samples = np.full((n_views, n_v, 3), np.nan, dtype=np.float32)
    for i in range(n_views):
        h, w = pointmaps[i].shape[:2]
        img_resized = np.array(images_list[i].resize((w, h)))

        world2cam = np.linalg.inv(cams2world[i])
        pts_cam = apply_transform(verts_raw, world2cam)
        z = pts_cam[:, 2]
        valid = z > 1e-3
        if not valid.any():
            continue

        f = float(np.atleast_1d(focals[i])[0])
        pp = np.atleast_1d(pps[i])
        cx, cy = (pp[0], pp[1]) if len(pp) >= 2 else (pp[0], pp[0])

        u = np.full(n_v, -1.0)
        v = np.full(n_v, -1.0)
        u[valid] = f * pts_cam[valid, 0] / z[valid] + cx
        v[valid] = f * pts_cam[valid, 1] / z[valid] + cy

        in_bounds = valid & (u >= 0) & (u < w - 1) & (v >= 0) & (v < h - 1)
        if not in_bounds.any():
            continue
        ui = u[in_bounds].astype(int)
        vi = v[in_bounds].astype(int)
        samples[i, in_bounds] = img_resized[vi, ui].astype(np.float32) / 255.0

    with np.errstate(invalid="ignore"):
        median_col = np.nanmedian(samples, axis=0)
    has_color = ~np.isnan(median_col).any(axis=1)
    return median_col, has_color


focals = scene.get_focals().detach().cpu().numpy()
pps = scene.get_principal_points().detach().cpu().numpy()

_bd_v = _backdrop.vertices
_proj_cols, _has_proj = reproject_colors(
    _bd_v, cams2world, focals, pps, pointmaps, images_inpainted_pil, ALIGN_INV)
print(f"재투영으로 색 채운 정점: {_has_proj.sum():,} / {len(_bd_v):,}")

# --- 폴백: 어느 뷰에서도 안 보이는 정점(구석 등)은 기존 방식(근처 실제 배경점 평균) 사용 ---
BD_COLOR_K = 3
_ref_all_pts, _ref_all_cols = all_pts[_is_bg], all_cols[_is_bg]
_ref_y = _ref_all_pts[:, 1]
_is_floor_ref = _ref_y < (floor_peak + BD_FLOOR_BAND[1])
_floor_ref_pts, _floor_ref_cols = _ref_all_pts[_is_floor_ref], _ref_all_cols[_is_floor_ref]
_wall_ref_pts, _wall_ref_cols = _ref_all_pts[~_is_floor_ref], _ref_all_cols[~_is_floor_ref]


def _sample_colors_fallback(query_pts, ref_pts, ref_cols, fallback_pts, fallback_cols, k=BD_COLOR_K):
    if len(ref_pts) < k:
        ref_pts, ref_cols = fallback_pts, fallback_cols
    tree = cKDTree(ref_pts)
    kq = min(k, len(ref_pts))
    _, idx = tree.query(query_pts, k=kq)
    if kq == 1:
        idx = idx[:, None]
    return np.median(ref_cols[idx], axis=1)


_bd_cols = _proj_cols.copy()
if (~_has_proj).any():
    _missing = ~_has_proj
    _missing_is_floor = _bd_v[_missing, 1] < (floor_peak + BD_FLOOR_BAND[1])
    _fill = np.zeros((_missing.sum(), 3))
    if _missing_is_floor.any():
        _fill[_missing_is_floor] = _sample_colors_fallback(
            _bd_v[_missing][_missing_is_floor], _floor_ref_pts, _floor_ref_cols, _ref_all_pts, _ref_all_cols)
    if (~_missing_is_floor).any():
        _fill[~_missing_is_floor] = _sample_colors_fallback(
            _bd_v[_missing][~_missing_is_floor], _wall_ref_pts, _wall_ref_cols, _ref_all_pts, _ref_all_cols)
    _bd_cols[_missing] = _fill
    print(f"  폴백(근접점 평균)으로 채운 정점: {_missing.sum():,}개")

_bd_cols_rgba = np.concatenate([
    (np.clip(_bd_cols, 0, 1) * 255).astype(np.uint8),
    np.full((len(_bd_cols), 1), 255, dtype=np.uint8),
], axis=1)
_backdrop.visual.vertex_colors = _bd_cols_rgba

room_parts["backdrop"] = _backdrop
print(f"backdrop 완성: 정점 {len(_backdrop.vertices):,} / 면 {len(_backdrop.faces):,}")


## 10. GLB 저장 (가구별 노드 분리 + backdrop 포함)

In [ ]:
from scipy.spatial import cKDTree as _cKDTree

SIMPLIFY_RATIO = 0.3

def _simplify(m, ratio):
    if ratio >= 1.0 or len(m.faces) < 1000:
        return m
    try:
        import fast_simplification
        v, f = fast_simplification.simplify(
            np.asarray(m.vertices, np.float64), np.asarray(m.faces, np.int64),
            target_reduction=1.0 - ratio)
        new = trimesh.Trimesh(vertices=v, faces=f, process=False)
        if hasattr(m.visual, "vertex_colors") and len(m.visual.vertex_colors):
            _, idx = _cKDTree(m.vertices).query(v)
            new.visual.vertex_colors = m.visual.vertex_colors[idx]
        return new
    except Exception as e:
        print(f"  simplify 건너뜀({e})")
        return m


_scene_export = trimesh.Scene()
_furn_meta = []
for name, m in room_parts.items():
    if m is None:
        continue
    m_s = _simplify(m, SIMPLIFY_RATIO)
    node = name.replace(" ", "_")
    _scene_export.add_geometry(m_s, node_name=node, geom_name=node)
    if name not in ("room", "backdrop"):
        _furn_meta.append({"name": name, "node": node})

GLB_PATH = f"/content/{TARGET_FOLDER}_toggle.glb"
_scene_export.export(GLB_PATH)
print(f"저장 완료: {GLB_PATH} ({os.path.getsize(GLB_PATH)/1024/1024:.2f} MB)")
print(f"토글 가능한 가구: {[f['name'] for f in _furn_meta]}")


저장 완료: /content/room06_toggle.glb (30.33 MB)
토글 가능한 가구: ['bed', 'refrigerator']


## 11. Three.js 뷰어 (가구 On/Off 토글)

가구 체크박스를 끄면 그 자리에 있던 backdrop(LaMa 재투영 색상)이 드러난다. HTML 저장 + 다운로드 방식.

In [ ]:
_glb_b64 = base64.b64encode(open(GLB_PATH, "rb").read()).decode("ascii")
_furn_json = json.dumps(_furn_meta, ensure_ascii=False)

_HTML = r"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__ - 가구 토글</title>
<style>
  body{margin:0;overflow:hidden;background:#111;}
  .hud{
    position:absolute;z-index:10;
    background:rgba(15,15,30,0.92);
    border:1px solid #3a3a6a;border-radius:12px;
    padding:12px 14px;font-family:sans-serif;
    box-shadow:0 4px 14px rgba(0,0,0,0.45);
    color:#ddd;font-size:13px;width:200px;
  }
  .hud-title{font-size:11px;font-weight:bold;color:#8b8bd0;letter-spacing:1px;margin:2px 0 8px 0;}
  .row{display:flex;justify-content:space-between;align-items:center;margin:6px 0;gap:8px;cursor:pointer;}
  .row span{color:#ccc;white-space:nowrap;}
  .row input[type=checkbox]{accent-color:#6366f1;cursor:pointer;}
  .row input[type=range]{width:105px;accent-color:#6366f1;}
</style>
</head>
<body>
<div class="hud" id="hud-left" style="top:12px;left:12px;">
  <div class="hud-title">보기</div>
  <label class="row"><span>바닥 격자</span><input type="checkbox" id="grid-toggle"></label>
  <div class="row"><span>밝기</span><input type="range" id="bright-slider" min="0" max="200" value="100"></div>
</div>

<div class="hud" id="hud-right" style="top:12px;right:12px;">
  <div class="hud-title">가구 On/Off</div>
  <div id="furn-list"></div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/loaders/GLTFLoader.js"></script>

<script>
(function(){
  const GLB_B64="__GLB__";
  const FURN=__FURN__;

  const scene=new THREE.Scene();
  scene.background=new THREE.Color(0x111111);

  const camera=new THREE.PerspectiveCamera(60,innerWidth/innerHeight,0.01,1000);
  const renderer=new THREE.WebGLRenderer({antialias:true});
  renderer.setSize(innerWidth,innerHeight);
  document.body.appendChild(renderer.domElement);

  const controls=new THREE.OrbitControls(camera,renderer.domElement);
  controls.enableDamping=true;

  scene.add(new THREE.AmbientLight(0xffffff,1.0));

  let roomMats=[];
  let nodeByName={};
  let bbox=null;

  const bytes=Uint8Array.from(atob(GLB_B64),c=>c.charCodeAt(0));
  new THREE.GLTFLoader().parse(bytes.buffer,"",function(gltf){
    const root=gltf.scene;
    root.traverse(o=>{
      if(o.isMesh){
        o.material=new THREE.MeshBasicMaterial({vertexColors:true,side:THREE.DoubleSide});
        roomMats.push(o.material);
        nodeByName[o.name]=o;
      }
    });
    scene.add(root);

    bbox=new THREE.Box3().setFromObject(root);
    const c=bbox.getCenter(new THREE.Vector3());
    const size=bbox.getSize(new THREE.Vector3()).length();
    camera.position.set(c.x+size*0.55,c.y+size*0.45,c.z+size*0.55);
    controls.target.copy(c);
    controls.update();

    const grid=new THREE.GridHelper(size*1.4,28,0x444466,0x26263c);
    grid.position.set(c.x,bbox.min.y,c.z);
    grid.visible=false;
    scene.add(grid);
    document.getElementById("grid-toggle").onchange=e=>grid.visible=e.target.checked;

    // 가구 토글 체크박스 생성
    const listEl=document.getElementById("furn-list");
    FURN.forEach(f=>{
      const row=document.createElement("label");
      row.className="row";
      row.innerHTML=`<span>${f.name}</span><input type="checkbox" checked>`;
      const cb=row.querySelector("input");
      cb.onchange=()=>{
        const node=nodeByName[f.node];
        if(node) node.visible=cb.checked;
      };
      listEl.appendChild(row);
    });
  });

  document.getElementById("bright-slider").oninput=e=>{
    const v=e.target.value/100;
    roomMats.forEach(m=>m.color.setScalar(v));
  };

  addEventListener("resize",()=>{
    camera.aspect=innerWidth/innerHeight;
    camera.updateProjectionMatrix();
    renderer.setSize(innerWidth,innerHeight);
  });

  (function loop(){
    requestAnimationFrame(loop);
    controls.update();
    renderer.render(scene,camera);
  })();
})();
</script>
</body>
</html>"""

html = _HTML.replace("__GLB__", _glb_b64).replace("__FURN__", _furn_json).replace("__TITLE__", TARGET_FOLDER)

HTML_PATH = f"/content/{TARGET_FOLDER}_toggle_viewer.html"
with open(HTML_PATH, "w") as f:
    f.write(html)
print(f"뷰어 저장: {HTML_PATH} ({os.path.getsize(HTML_PATH)/1024/1024:.2f} MB)")

from google.colab import files
files.download(HTML_PATH)


뷰어 저장: /content/room06_toggle_viewer.html (40.45 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>